# TGI / vLLM / llama.cppによる推論効率化

Transformersには、本番環境での推論を最適化するための以下のフレームワークが準備されています。

- TGI: Flash Attentionの仕組みを用いることで、推論を高速化
- vLLM: KVキャッシュによる推論高速化を、PagedAttentionによりVRAMを節約しながら実現
- llama.cpp: 

## TGI

TGI（Text Generation Inference）は、Flash Attentionの仕組みを
Flash Attentionとは、GPU内のHBM（大容量だが転送速度が遅い）とSRAM（超高速だが容量小さい）という2種類のメモリ領域のうち、以下の手順でSRAM内でAttention演算を極力完結させて、HBM転送のボトルネックを防ぐ手法です。

- HBMにQKVをロード
- ソフトマックス計算に用いるゼロ行列$O$、および$l$を作る
- for j（ブロック$j$の外側ループ）
    - KVをSRAM内に収まる小さいブロック$K_j, V_j$に分割してSRAMに送る（タイリング）
    - for i（ブロック$i$の内側ループ）
        - Qのブロック$i$、ソフトマックス計算の途中経過$O_i$
- 

## vLLM

vLLMは、LLMの推論高速化に用いられるKVキャッシュの弱点であるVRAMの大量使用を、PagedAttentionと呼ばれる方法によるVRAMの効率利用で軽減できるフレームワークです。

1. メモリページング：KVキャッシュを1つの大きなブロックとして扱うのではなく、固定サイズの「ページ」（オペレーティングシステムの仮想メモリに類似）に分割します。
2. 非連続ストレージ：ページはGPUメモリ内で連続して格納する必要がないため、より柔軟なメモリ割り当てが可能になります。
3. ページテーブル管理：ページテーブルは、どのページがどのシーケンスに属するかを追跡し、効率的な検索とアクセスを可能にします。
4. メモリ共有：並列サンプリングなどの操作では、プロンプトのKVキャッシュを格納するページを複数のシーケンス間で共有できます。